In [1]:
import janitor
import seawater as sw
from erddapy import ERDDAP


def clean_ra_df(df):
    cols = [
        "station_id",
        "station_description",
        "station_long_name",
        "wmo_id_or_nws_cman_id",
        "station_description",
        "latitude_dec_deg",
        "longitude_dec_deg",
    ]

    df = sheets[ra].clean_names(
        remove_special=True,
        strip_underscores=True
    )[cols].remove_empty().rename(
        columns={"latitude_dec_deg": "lat", "longitude_dec_deg": "lon"}
    ).fillna("")
    
    df["lon"] = df["lon"].astype(float)
    df["lat"] = df["lat"].astype(float)
    return df


def clean_server_df(urls, ra):
    server = urls[ra]
    e = ERDDAP(
        server=server,
        protocol="tabledap",
    )
    e.dataset_id = "allDatasets"
    df = e.to_pandas()
    pos = ["minLongitude (degrees_east)", "minLatitude (degrees_north)"]
    df[pos] = df[pos].astype(float)
    return df


def match_positions(df_server, df_ra):
    res = {}
    cols = ["datasetID", "institution"]
    for k, station in df_ra.iterrows():
        matches = []
        for j, row in df_server.iterrows():
            dist, _ = sw.dist(
                lat=[station["lat"], row["minLatitude (degrees_north)"]],
                lon=[station["lon"], row["minLongitude (degrees_east)"]]
            )
            # Check if station less 500 meters away.
            if dist < 0.5:
                matches.append(df_server.iloc[j][cols].tolist())
        res.update({station["station_long_name"]: matches})
    return res

/tmp/ipykernel_145026/1219850833.py:2: UserWarning: The seawater library is deprecated! Please use gsw instead.
  import seawater as sw


In [2]:
import json


with open("utils/ra_erddaps.json") as f:
    urls = json.load(f)

urls

{'aoos': 'http://erddap.aoos.org/erddap',
 'glos': 'https://seagull-erddap.glos.org/erddap',
 'glos2': 'http://data.glos.us/erddap',
 'gcoos': 'https://erddap.gcoos.org/erddap',
 'gcoos2': 'https://gcoos5.geos.tamu.edu/erddap',
 'secoora': 'http://erddap.secoora.org/erddap',
 'maracoos': 'https://erddap.maracoos.org/erddap',
 'pacioos': 'https://pae-paha.pacioos.hawaii.edu/erddap',
 'pacioos2': 'http://oos.soest.hawaii.edu/erddap',
 'cencoos': 'http://erddap.cencoos.org/erddap',
 'nanoos': 'http://data.nanoos.org/erddap',
 'caricoos': 'http://dm3.caricoos.org:8002/erddap',
 'neracoos': 'https://data.neracoos.org/erddap',
 'neracoos2': 'http://www.neracoos.org/erddap',
 'sccoos': 'https://erddap.sccoos.org/erddap'}

In [3]:
from pathlib import Path
import pprint

import pandas as pd


sheets = {}
for fname in Path("2024/data/processed/").glob("*.xlsx"):
    sheets.update({fname.stem.lower(): pd.read_excel(fname)})

In [4]:
for ra, df_ra in sheets.items():
    print(f"{ra=}")
    df_ra = clean_ra_df(df_ra)
    try:
        df_server = clean_server_df(urls, ra)
    except Exception as e:
        print(f"Cannot access ERDDAP server for {ra}.\n{e}")
        continue
    res = match_positions(df_server, df_ra)
    pprint.pprint(res)

ra='pacioos'
{'Ala Wai Buoy': [],
 'Aunuu, American Samoa': [],
 'Coconut Island Weather Station': [['aws_himb',
                                     'Pacific Islands Ocean Observing System '
                                     '(PacIOOS)']],
 'Ebiil, Palau PP': [],
 'Fagatele Bay, Tutuila, Am Samoa': [],
 'Hanalei, Kauai,  Hawaii': [],
 'Hawaii Yacht Club NSS': [['nss_001',
                            'Pacific Islands Ocean Observing System (PacIOOS)'],
                           ['nss_002',
                            'Pacific Islands Ocean Observing System '
                            '(PacIOOS)']],
 'Hilo Bay, Hawaii': [['wqb_04',
                       'Pacific Islands Ocean Observing System (PacIOOS)']],
 'Hilo, Hawaii': [],
 'Hilton Hawiian Village NSS': [['nss_003',
                                 'Pacific Islands Ocean Observing System '
                                 '(PacIOOS)']],
 'Ipan, Guam': [],
 'Kalaeloa, Barbers Point, HI': [],
 'Kalama Beach Park, Maui NSS': [['

{'Bodega': [['bodega-head-intertidal-shore-sta',
             'San Francisco State University (SFSU)'],
            ['bodega-marine-laboratory-weather',
             'University of California Davis, Bodega Marine Laboratory'],
            ['ncei-uscrn-ca_bodega_6_wsw-v2',
             'US Climate Research Network (USCRN, NOAA)'],
            ['bodega-bay-bml_wts', 'Bodega Marine Laboratory']],
 'Bodega IFCB163': [['bodega-head-intertidal-shore-sta',
                     'San Francisco State University (SFSU)'],
                    ['bodega-marine-laboratory-weather',
                     'University of California Davis, Bodega Marine '
                     'Laboratory'],
                    ['ncei-uscrn-ca_bodega_6_wsw-v2',
                     'US Climate Research Network (USCRN, NOAA)'],
                    ['bodega-bay-bml_wts', 'Bodega Marine Laboratory']],
 'Cal Poly Humboldt IFCB': [['humboldt-bay-burkeolator',
                             'Central & Northern California Ocean Obs

{'Aripeka': [['edu_usf_marine_comps_arpf1',
              'University of South Florida Coastal Ocean Monitoring and '
              'Prediction System (COMPS)']],
 'Bald Head Island': [['hohonu_10057_bald_head_island', 'Hohonu']],
 "CSI-Jennette'sSpotter1": [['org_secoora_rstsnca_jennettes_pi',
                             'South-East Zoo Alliance for Reproduction and '
                             'Conservation (SEZARC)']],
 'Cape Hatteras East': [['250-cape-hatteras-east-nc',
                         'Coastal Data Information Program (CDIP)']],
 'Cape Lookout National Seashore': [['hohonu_10048_cape_lookout_nation',
                                     'Hohonu']],
 'Capers Nearshore': [],
 'Capers Nearshore Wave': [],
 'Captiva Island, FL': [['hohonu_10036_captiva_island_fl', 'Hohonu']],
 'Charleston 60 ': [],
 'Charleston 60 Wave': [],
 'City of Beaufort SC': [['hohonu_10047_city_of_beaufort_sc', 'Hohonu']],
 'Colington Creek Inn Dock, Kill Devil Hills, NC': [['collington-creek-inn-

{'Agua Hedionda Lagoon near Carlsbad Aquafarm': [],
 'Bodega Marine Lab': [['HABs-BodegaMarineLab', 'CalHABMAP']],
 'Cal Poly Pier': [['HABs-CalPolyPier', 'CalHABMAP']],
 'Catalina Sea Ranch - NOMAD Buoy ': [],
 'Del Mar Mooring': [],
 'Del Mar Mooring ': [],
 'Newport Beach Pier': [['HABs-NewportBeachPier', 'CalHABMAP'],
                        ['SPATT-NewportBeachPier', 'CalHABMAP']],
 'Observing nutrient fluxes and their role in HAB development in the nearshore region of Southern California. ': [],
 'Power Buoy/M1': [],
 'San Onofre': [],
 'Santa Cruz Wharf ': [],
 'Santa Monica Pier': [['HABs-SantaMonicaPier', 'CalHABMAP']],
 'Scripps Pier': [['HABs-ScrippsPier', 'CalHABMAP'],
                  ['SPATT-ScrippsPier', 'CalHABMAP']],
 'Stearns Wharf': [['HABs-StearnsWharf', 'CalHABMAP'],
                   ['SPATT-StearnsWharf', 'CalHABMAP']],
 'Trinidad Pier/Hog Island': [['HABs-TrinidadPier', 'CalHABMAP'],
                              ['SPATT-TrinidadPier', 'CalHABMAP']],
 'off SIO

{'003: Rincon del San Jose (87778121): Rincon del San Jose; Potrero Lopeno SW, TX ': [],
 '005: Packery Channel (87757921): Packery Channel, TX': [],
 '006: Ingleside (87752831): Port Ingleside, TX ': [],
 '009: Port Aransas (87752371): Port Aransas, TX': [],
 '013: S. Bird Island (87761391): South Bird Island, TX': [['CBI_171',
                                                            'GCOOS']],
 '031: Seadrift (87730371): Seadrift, TX': [['CBI_130', 'GCOOS']],
 '041: Nueces Delta 1: Nueces Delta 1': [],
 '042: Nueces Delta 2: Nueces Delta 2': [['CBI_042', 'GCOOS']],
 '043: Nueces Delta 3: Nueces Delta 3': [],
 "057: Port O'Connor (87737011): Matagorda Bay; Port O'Connor, TX": [],
 '068: Baffin Bay (87766041): Baffin Bay; Point of Rocks, TX': [['CBI_170',
                                                                 'GCOOS']],
 '069: Nueces Delta Weather Station': [['CBI_069', 'GCOOS']],
 '072: SALT01 (Nueces Bay, Texas): SALT01 (Nueces Bay, Texas)': [['CBI_072',
                

Cannot access ERDDAP server for nanoos.
[Errno 111] Connection refused
